# DH-PINN plate bending under SINUSOIDAL loading, FULLY-CLAMPED on all four edges (FFFF case).

In [1]:
"""
DH-PINN (Deterministic-Harmonic + Vanilla-PINN backbone) for Kirchhoff-Love
plate bending under SINUSOIDAL loading, FULLY-CLAMPED on all four edges
(FFFF case).

Direct extension of dh_pinn_uniform_hardBC_FFFF.py (the FFFF/uniform-load
script), which itself carries the FD-reference-solver and squared-envelope
machinery built for dh_pinn_triangular_hardBC_FFFF.py. Backbone, training
loop, optimizer/schedule, diagnostics, and plotting layout are carried over
unchanged. Effectively this file is "the sinusoidal load, but with the
clamped-plate BC/solver apparatus" -- same load shape as the original
SS-case sinusoidal script, same BC/solver apparatus as the other FFFF
scripts.

-----------------------------------------------------------------------------
WHAT CHANGES FOR FFFF + SINUSOIDAL LOAD (read before trusting results)
-----------------------------------------------------------------------------
1. THE LOAD. q(x,y) = q0 * sin(pi*x/a) * sin(pi*y/b) -- a single half-sine
   hump over the whole plate, zero on all four edges and maximum at the
   center. Replaces the constant q(x,y) = q0 of the uniform-load FFFF
   script. `sinusoidal_load_nd` replaces `uniform_load_nd` in the PDE
   residual and in the FD reference solve's RHS.

2. THE BOUNDARY CONDITION -- FFFF (fully clamped, all four edges).
   Clamped edges impose TWO essential (Dirichlet-type) conditions per edge:
       w = 0            (no deflection)
       dw/dn = 0        (no rotation / zero slope normal to the edge)
   Both conditions are now HARD-enforced through the network's output
   envelope (see #3). There is no natural (moment) boundary condition left
   to enforce softly -- clamped edges do not have Mx=0 or My=0 the way
   simply-supported edges do. Consequently the `moment_loss` term from the
   SS script is REMOVED entirely; `total_loss = pde_loss` only. The old
   soft moment-BC diagnostic is replaced by a soft slope-BC diagnostic
   (`slope_bc_loss`, tracking d w/dn at the four edges) which -- like the
   deflection diagnostic -- should sit at ~0 purely as a consequence of the
   envelope construction, not because it is optimized.

3. THE HARD-BC ENVELOPE IS SQUARED. For SS-SS-SS-SS, envelope =
   xi(1-xi)*eta(1-eta) is enough: it vanishes at every edge, which is the
   only essential condition there (moment=0 was handled as a soft penalty).
   For FFFF, an edge factor of the form xi(1-xi) still vanishes at xi=0,1,
   but its *derivative* does NOT vanish there -- so slope would be
   unconstrained. Squaring each edge factor fixes this: d/dxi[(xi(1-xi))^2]
   = 2*xi(1-xi)*(1-2xi), which is ALSO exactly zero at xi=0 and xi=1. So

       envelope = [xi(1-xi)]^2 * [eta(1-eta)]^2

   hard-enforces w=0 AND dw/dn=0 on all four edges simultaneously, for any
   raw network output -- no loss term needed to teach either condition.

4. THE ANALYTICAL BENCHMARK IS A FINITE-DIFFERENCE BIHARMONIC SOLVE, NOT A
   NAVIER SERIES. For the SS-SS-SS-SS version of this exact load, a
   sinusoidal q(x,y) = q0 sin(pi x/a) sin(pi y/b) is already a single Navier
   mode, so the SS case had a closed-form ONE-TERM solution (no truncated
   series needed at all -- see the original sinusoidal SS script). That
   convenience is entirely a consequence of sin(m pi x/a) being the SS
   eigenfunction; it does NOT carry over to a clamped plate, because
   sin(pi x/a) sin(pi y/b) is not an eigenfunction of the clamped biharmonic
   problem (it satisfies w=0 on the edges but NOT dw/dn=0). So even though
   the load is the simplest possible single-mode shape, there is still no
   closed form here, and the reference solution is obtained the same way as
   in the other FFFF scripts: directly solving
       D * (w_xxxx + 2 w_xxyy + w_yyyy) = q(x,y)
   with a standard second-order finite-difference discretization on the
   SAME (nx, ny) grid used for PINN collocation, with clamped edges
   enforced via the classical ghost-point mirror trick:
       dw/dn = 0 at an edge  <=>  w_ghost = w_(one node inside the edge)
   (central difference of the first derivative at the edge node is zero
   exactly when the ghost node mirrors the first interior node). Combined
   with w=0 exactly on the boundary itself, this eliminates all ghost
   unknowns and leaves a sparse linear system in the interior nodes only,
   solved once with `scipy.sparse.linalg.spsolve`. This FD solve is a
   ONE-OFF cost (not used inside the training loop) and its accuracy is
   controlled by (nx, ny) -- refine the grid if tighter benchmark accuracy
   is needed. The solver itself was independently sanity-checked against
   the textbook clamped-square-plate-under-uniform-load center-deflection
   coefficient (~0.00126 q0 a^4/D, Timoshenko & Woinowsky-Krieger, see the
   uniform-load FFFF script); the discretization is unchanged here, only
   the RHS load differs. Mx, My, Mxy for the reference are obtained by
   differentiating the FD solution field with (separate, second-order)
   central differences.

5. THE DH-PINN EMBEDDING'S HARMONICS. Unlike the uniform-load and
   triangular-load FFFF scripts, this one needs no change in harmonic
   *selection philosophy* from the original SS/sinusoidal script: that
   script already used CONSECUTIVE indices 1..K (not odd-only -- the
   odd-only restriction was specific to the uniform load's Fourier
   content, see the uniform-load scripts' docstrings), precisely because a
   single sin(pi x/a) sin(pi y/b) mode doesn't motivate truncating to a
   subset of harmonics in the first place. So `HARMONIC_INDICES =
   [1, 2, 3, 4, 5]` here is consistent with both (a) the SS/sinusoidal
   script's original choice and (b) the general "no proven parity for a
   clamped plate" reasoning used in the other FFFF scripts.
   `USE_PRODUCT_TERMS` (modal sin*sin features) is kept for the same
   reason as before -- a reasonable smooth basis for a doubly-varying
   field, and in this case the load itself is literally one of the modal
   product terms (m=n=1), so the network should pick it up quickly.

Everything else -- backbone width/depth/activation, optimizer, LR schedule,
gradient clipping, epoch-level cost/error logging, and the plotting suite --
is structurally identical to dh_pinn_uniform_hardBC.py.

Outputs (all written to ./results/<MODEL_NAME>/):
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python dh_pinn_sinusoidal_hardBC_FFFF.py
"""

import os
import json
import time
import random

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "DH_PINN_Modal_HardBC_FFFF_Sinusoidal"  # >>> VARIANT POINT
LOADING_CASE = "sinusoidal"
BC_CASE = "FFFF"  # fully clamped, all four edges

# --- DH-PINN specific config -------------------------------------------
HARMONIC_INDICES = [1, 2, 3, 4, 5]  # consecutive -- matches the original
                                     # SS/sinusoidal script's choice; no
                                     # Navier basis exists to restrict
                                     # parity against for a clamped plate.
INCLUDE_RAW_COORDS = True   # concatenate raw (xi, eta) alongside harmonics
USE_PRODUCT_TERMS = True    # add sin(m*pi*xi)*sin(n*pi*eta) modal features
                             # for m,n in HARMONIC_INDICES x HARMONIC_INDICES
GRAD_CLIP_NORM = 1.0        # global grad-norm clip; None to disable
EMBEDDING_TYPE = "deterministic_harmonic_modal_full" if USE_PRODUCT_TERMS else "deterministic_harmonic_full"

# --- FD reference-solver config (replaces the SS-case Navier truncation) --
# Uses the SAME (nx, ny) grid as the PINN collocation grid (defined below)
# so predicted and reference fields line up node-for-node. Refine nx, ny
# for a tighter benchmark; this is a one-off sparse solve, not part of
# training.

OUT_DIR = os.path.join("results", MODEL_NAME)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 15000
PATIENCE = 3000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0  # real_q0 = peak (center) sinusoidal load intensity

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

# ============================================================
# DOMAIN
# ============================================================
nx, ny = 51, 51
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

X_grid = torch.tensor(grid_points_nd, dtype=DTYPE, device=DEVICE)

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept only for the (diagnostic) deflection/slope
#     residual now, since both essential conditions are HARD-enforced. ----
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)

bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)

# --- per-edge indices, now used only for the SLOPE diagnostic (no soft
#     moment BC exists for a clamped edge) -----------------------------
edge_x0 = np.where(grid_points_nd[:, 0] == 0)[0]
edge_xa = np.where(grid_points_nd[:, 0] == a_nd)[0]
edge_y0 = np.where(grid_points_nd[:, 1] == 0)[0]
edge_yb = np.where(grid_points_nd[:, 1] == b_nd)[0]

edge_x_indices = np.unique(np.concatenate([edge_x0, edge_xa]))  # dw/dx = 0 here
edge_y_indices = np.unique(np.concatenate([edge_y0, edge_yb]))  # dw/dy = 0 here

edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)


def sinusoidal_load_nd(x, y):
    """q(x,y) = q0 * sin(pi*x/a) * sin(pi*y/b): single half-sine hump, zero
    on all four edges, maximum at the plate center."""
    return q0_nd * torch.sin(np.pi * x / a_nd) * torch.sin(np.pi * y / b_nd)


# ============================================================
# FINITE-DIFFERENCE BIHARMONIC REFERENCE SOLVER (clamped, all edges)
# ============================================================
#   D (w_xxxx + 2 w_xxyy + w_yyyy) = q(x,y),   w = dw/dn = 0 on boundary
#
# Discretization (second-order central differences, general hx, hy):
#   w_xxxx  ~ [w(i-2,j) -4w(i-1,j) +6w(i,j) -4w(i+1,j) +w(i+2,j)] / hx^4
#   w_yyyy  ~ [w(i,j-2) -4w(i,j-1) +6w(i,j) -4w(i,j+1) +w(i,j+2)] / hy^4
#   w_xxyy  ~ [ (w(i+1,j+1)-2w(i,j+1)+w(i-1,j+1))
#               -2*(w(i+1,j)  -2w(i,j)  +w(i-1,j))
#               + (w(i+1,j-1)-2w(i,j-1)+w(i-1,j-1)) ] / (hx^2 hy^2)
#
# Clamped edge -> ghost-point mirror (central-difference slope = 0):
#   w(-1,j) = w(1,j),          w(nx,j) = w(nx-2,j)
#   w(i,-1) = w(i,1),          w(i,ny) = w(i,ny-2)
# and w = 0 exactly on the boundary nodes themselves (i=0, i=nx-1, j=0,
# j=ny-1), so only interior nodes (1..nx-2, 1..ny-2) are unknowns.
def _biharmonic_stencil(hx, hy):
    """Returns {(di, dj): coeff} for D*(w_xxxx + 2 w_xxyy + w_yyyy)."""
    coeffs = {}

    def add(di, dj, c):
        coeffs[(di, dj)] = coeffs.get((di, dj), 0.0) + c

    # w_xxxx / hx^4
    for di, c in [(-2, 1.0), (-1, -4.0), (0, 6.0), (1, -4.0), (2, 1.0)]:
        add(di, 0, c / hx ** 4)
    # w_yyyy / hy^4
    for dj, c in [(-2, 1.0), (-1, -4.0), (0, 6.0), (1, -4.0), (2, 1.0)]:
        add(0, dj, c / hy ** 4)
    # 2 * w_xxyy / (hx^2 hy^2)
    mixed = {
        (1, 1): 1.0, (0, 1): -2.0, (-1, 1): 1.0,
        (1, 0): -2.0, (0, 0): 4.0, (-1, 0): -2.0,
        (1, -1): 1.0, (0, -1): -2.0, (-1, -1): 1.0,
    }
    for (di, dj), c in mixed.items():
        add(di, dj, 2.0 * c / (hx ** 2 * hy ** 2))
    return coeffs


def _resolve(i, n):
    """Clamped ghost-mirror index resolution along one axis."""
    if i == -1:
        return 1
    if i == n:
        return n - 2
    return i  # 0..n-1, possibly a boundary node (value 0)


def solve_clamped_plate_fd(q_grid, nx, ny, hx, hy, D):
    """Solve D*nabla^4 w = q for a fully-clamped rectangular plate.

    q_grid: (ny, nx) array of load values at grid nodes (matches X_nd/Y_nd
    'xy' meshgrid convention: q_grid[j, i] <-> (x_i, y_j)).
    Returns w_grid: (ny, nx) array, exactly 0 on the boundary.
    """
    coeffs = _biharmonic_stencil(hx, hy)

    interior_ij = [(i, j) for j in range(1, ny - 1) for i in range(1, nx - 1)]
    idx_map = {ij: k for k, ij in enumerate(interior_ij)}
    n_unknown = len(interior_ij)

    A = sp.lil_matrix((n_unknown, n_unknown))
    b = np.zeros(n_unknown)

    for (i, j), row in idx_map.items():
        b[row] = q_grid[j, i] / D
        for (di, dj), c in coeffs.items():
            ii = _resolve(i + di, nx)
            jj = _resolve(j + dj, ny)
            if ii == 0 or ii == nx - 1 or jj == 0 or jj == ny - 1:
                continue  # boundary node, value is exactly 0 -- no contribution
            col = idx_map[(ii, jj)]
            A[row, col] += c

    w_interior = spla.spsolve(A.tocsr(), b)

    w_grid = np.zeros((ny, nx))
    for (i, j), row in idx_map.items():
        w_grid[j, i] = w_interior[row]
    return w_grid


def _central_diff_moments(w_grid, hx, hy, D, nu):
    """Mx, My, Mxy from a nodal deflection field via 2nd-order central
    differences (one-sided at the boundary rows/cols)."""
    w_x = np.gradient(w_grid, x_nd, axis=1)
    w_y = np.gradient(w_grid, y_nd, axis=0)
    w_xx = np.gradient(w_x, x_nd, axis=1)
    w_yy = np.gradient(w_y, y_nd, axis=0)
    w_xy = np.gradient(w_x, y_nd, axis=0)

    Mx = -D * (w_xx + nu * w_yy)
    My = -D * (w_yy + nu * w_xx)
    Mxy = -(1 - nu) * D * w_xy
    return Mx, My, Mxy


def _fd_reference_fields():
    hx = a_nd / (nx - 1)
    hy = b_nd / (ny - 1)
    q_grid_nd = q0_nd * np.sin(np.pi * X_nd / a_nd) * np.sin(np.pi * Y_nd / b_nd)  # (ny, nx), matches sinusoidal_load_nd
    w = solve_clamped_plate_fd(q_grid_nd, nx, ny, hx, hy, D_nd)
    Mx, My, Mxy = _central_diff_moments(w, hx, hy, D_nd, nu_nd)
    return w, Mx, My, Mxy


# ============================================================
# DETERMINISTIC HARMONIC EMBEDDING  (>>> the DH-PINN input feature block)
# ============================================================
class HarmonicEmbedding(nn.Module):
    """
    Deterministic sinusoidal (harmonic) input feature transformation.

    gamma(xi, eta) = [xi, eta]  (optional, see INCLUDE_RAW_COORDS)
                      ++ [sin(k*pi*xi), cos(k*pi*xi)]_{k in harmonic_indices}   (per-axis)
                      ++ [sin(k*pi*eta), cos(k*pi*eta)]_{k in harmonic_indices} (per-axis)
                      ++ [sin(m*pi*xi)*sin(n*pi*eta)]_{m,n in harmonic_indices} (modal / product,
                                                                                  see USE_PRODUCT_TERMS)

    `harmonic_indices` is CONSECUTIVE here (1..K), not the odd-only set used
    for the SS/uniform-load script -- that odd-only restriction encoded a
    property of the SS Navier series, which has no counterpart for a
    clamped plate (no Navier basis exists regardless of load, see module
    docstring point #4). So the harmonics are used as a generic
    deterministic Fourier feature basis rather than an encoding of a
    proven symmetry (see module docstring, point #5).
    """

    def __init__(self, harmonic_indices, include_raw_coords: bool = True,
                 use_product_terms: bool = True):
        super().__init__()
        self.harmonic_indices = list(harmonic_indices)
        n_harmonics = len(self.harmonic_indices)
        self.include_raw_coords = include_raw_coords
        self.use_product_terms = use_product_terms
        k = torch.tensor(self.harmonic_indices, dtype=DTYPE, device=DEVICE).view(1, -1)
        self.register_buffer("k", k)

        out_dim = (2 if include_raw_coords else 0) + 4 * n_harmonics
        if use_product_terms:
            out_dim += n_harmonics * n_harmonics  # KxK grid of (m,n) modes
        self.out_dim = out_dim

    def forward(self, xi, eta):
        kx = self.k * np.pi * xi   # (N, K), broadcasts (N,1) * (1,K)
        ky = self.k * np.pi * eta
        sin_x, cos_x = torch.sin(kx), torch.cos(kx)   # (N, K) each
        sin_y, cos_y = torch.sin(ky), torch.cos(ky)

        feats = [sin_x, cos_x, sin_y, cos_y]
        if self.include_raw_coords:
            feats = [xi, eta] + feats
        if self.use_product_terms:
            # outer product sin_x[:,m] * sin_y[:,n] for every (m,n) pair,
            # flattened to (N, K*K) -- this is the modal feature basis.
            modal = sin_x.unsqueeze(2) * sin_y.unsqueeze(1)  # (N, K, K)
            feats.append(modal.reshape(modal.shape[0], -1))
        return torch.cat(feats, dim=1)


# ============================================================
# MODEL   (backbone identical to the SS/uniform script: SiLU MLP;
#          only the input block AND the hard-BC envelope differ -- the
#          envelope is now SQUARED to hard-enforce w=0 AND dw/dn=0, see
#          docstring point #3)
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3,
                 harmonic_indices=HARMONIC_INDICES, include_raw_coords=INCLUDE_RAW_COORDS,
                 use_product_terms=USE_PRODUCT_TERMS):
        super().__init__()
        self.embed = HarmonicEmbedding(harmonic_indices, include_raw_coords, use_product_terms)

        layers = []
        in_dim = self.embed.out_dim   # <-- only structural diff vs. vanilla (was 2)
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def forward(self, x, y):
        xi = x / a_nd
        eta = y / b_nd

        h = self.embed(xi, eta)
        h = self.hidden(h)
        raw = self.out(h)

        # Hard-constrained clamped BC (w = 0 AND dw/dn = 0 on all edges).
        # SQUARING each edge factor -- vs. the single power used for the
        # SS case -- makes both the envelope AND its first derivative
        # vanish exactly at every edge, for any raw network output.
        edge_x = xi * (1 - xi)
        edge_y = eta * (1 - eta)
        envelope = (edge_x ** 2) * (edge_y ** 2)
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS (4th-order derivatives via autograd)
#
# NOTE vs. the SS/uniform script: there is no moment_loss term. Both
# essential clamped conditions (w=0, dw/dn=0) are hard-enforced by the
# envelope, so total_loss = pde_loss only. bc_loss and slope_bc_loss are
# retained purely as diagnostics (expected ~0 by construction).
# ============================================================
def compute_loss():
    x = X_grid[:, 0:1].clone().requires_grad_(True)
    y = X_grid[:, 1:2].clone().requires_grad_(True)

    w = model(x, y)
    ones = torch.ones_like(w)

    w_x = torch.autograd.grad(w, x, grad_outputs=ones, create_graph=True)[0]
    w_y = torch.autograd.grad(w, y, grad_outputs=ones, create_graph=True)[0]

    w_xx = torch.autograd.grad(w_x, x, grad_outputs=ones, create_graph=True)[0]
    w_yy = torch.autograd.grad(w_y, y, grad_outputs=ones, create_graph=True)[0]

    w_xxx = torch.autograd.grad(w_xx, x, grad_outputs=ones, create_graph=True)[0]
    w_yyy = torch.autograd.grad(w_yy, y, grad_outputs=ones, create_graph=True)[0]
    w_xxy = torch.autograd.grad(w_xx, y, grad_outputs=ones, create_graph=True)[0]

    w_xxxx = torch.autograd.grad(w_xxx, x, grad_outputs=ones, create_graph=True)[0]
    w_yyyy = torch.autograd.grad(w_yyy, y, grad_outputs=ones, create_graph=True)[0]
    w_xxyy = torch.autograd.grad(w_xxy, y, grad_outputs=ones, create_graph=True)[0]

    q = sinusoidal_load_nd(x, y)

    pde = (w_xxxx + 2 * w_xxyy + w_yyyy - q)
    pde_loss = torch.mean(pde[int_idx] ** 2)

    # DIAGNOSTICS ONLY -- should be ~0 exactly, by construction of the
    # squared envelope. Neither contributes to `total` (no gradient path
    # into the optimizer beyond what pde_loss already provides).
    bc_loss = torch.mean(w[bc_idx] ** 2)
    slope_bc_loss = (torch.mean(w_x[edge_x_idx] ** 2) +
                      torch.mean(w_y[edge_y_idx] ** 2))

    total = pde_loss
    return total, pde_loss, bc_loss, slope_bc_loss, w


# ============================================================
# TRAINING LOOP with computational-cost logging + early stopping
# ============================================================
epoch_history, loss_history, pde_history, bc_history, slope_bc_history = [], [], [], [], []
w_center_history, epoch_time_history = [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc, slope_bc, w_pred = compute_loss()
    total.backward()
    if GRAD_CLIP_NORM is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    slope_bc_history.append(slope_bc.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss {loss_val:.6e} (pde {pde.item():.3e}) "
              f"hardBC_residual {bc.item():.3e} slope_residual {slope_bc.item():.3e} "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    if loss_val < best_loss:
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            converged_epoch = epoch
            break

total_train_time = time.perf_counter() - train_start
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_bc_residual_diagnostic": bc_history,
    "slope_bc_residual_diagnostic": slope_bc_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS BALANCE DIAGNOSTIC
# ============================================================
final_pde, final_bc, final_slope_bc = pde_history[-1], bc_history[-1], slope_bc_history[-1]
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "embedding_type": EMBEDDING_TYPE,
    "harmonic_indices": HARMONIC_INDICES,
    "include_raw_coords": INCLUDE_RAW_COORDS,
    "use_product_terms": USE_PRODUCT_TERMS,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "final_pde_loss": final_pde,
    "final_hard_bc_residual_diagnostic": final_bc,
    "final_slope_bc_residual_diagnostic": final_slope_bc,
    "note": "total_loss == pde_loss for FFFF; both essential BCs are hard-enforced "
            "by the squared envelope, so there is no moment/soft-BC term to balance.",
}
if final_bc > 1e-20 or final_slope_bc > 1e-20:
    print(f"WARNING: hard_bc_residual_diagnostic = {final_bc:.3e}, "
          f"slope_bc_residual_diagnostic = {final_slope_bc:.3e} -- expected ~0 -- "
          f"double-check the envelope function / grid point exactness (float64 expected).")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "embedding_type": EMBEDDING_TYPE,
    "harmonic_indices": HARMONIC_INDICES,
    "include_raw_coords": INCLUDE_RAW_COORDS,
    "use_product_terms": USE_PRODUCT_TERMS,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "input_feature_dim": model.embed.out_dim,
    "device": str(DEVICE),
    "epochs_run": len(epoch_history),
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,  # deprecated alias, same value
    "mean_time_per_epoch_s": float(np.mean(epoch_time_history)),
    "median_time_per_epoch_s": float(np.median(epoch_time_history)),
    "total_training_time_s": total_train_time,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My, Mxy) via autograd
# ============================================================
def compute_full_fields():
    x = X_grid[:, 0:1].clone().requires_grad_(True)
    y = X_grid[:, 1:2].clone().requires_grad_(True)
    w = model(x, y)
    ones = torch.ones_like(w)

    w_x = torch.autograd.grad(w, x, grad_outputs=ones, create_graph=True)[0]
    w_y = torch.autograd.grad(w, y, grad_outputs=ones, create_graph=True)[0]
    w_xx = torch.autograd.grad(w_x, x, grad_outputs=ones, create_graph=True)[0]
    w_yy = torch.autograd.grad(w_y, y, grad_outputs=ones, create_graph=True)[0]
    w_xy = torch.autograd.grad(w_x, y, grad_outputs=ones, create_graph=True)[0]

    Mx = -(w_xx + nu_nd * w_yy)
    My = -(w_yy + nu_nd * w_xx)
    Mxy = -(1 - nu_nd) * w_xy

    w_grid = w.detach().cpu().numpy().reshape(ny, nx)
    Mx_grid = Mx.detach().cpu().numpy().reshape(ny, nx)
    My_grid = My.detach().cpu().numpy().reshape(ny, nx)
    Mxy_grid = Mxy.detach().cpu().numpy().reshape(ny, nx)
    return w_grid, Mx_grid, My_grid, Mxy_grid


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd = compute_full_fields()

w_anal_nd, Mx_anal_nd, My_anal_nd, Mxy_anal_nd = _fd_reference_fields()

w_dim_pred, w_dim_anal = w_pred_nd * W, w_anal_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_anal = Mx_pred_nd * moment_scale, Mx_anal_nd * moment_scale
My_dim_pred, My_dim_anal = My_pred_nd * moment_scale, My_anal_nd * moment_scale
Mxy_dim_pred, Mxy_dim_anal = Mxy_pred_nd * moment_scale, Mxy_anal_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_anal
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_anal)
My_error = np.abs(My_dim_pred - My_dim_anal)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_anal)

# ============================================================
# ERROR METRICS
# ============================================================
# For this sinusoidal load on a clamped SQUARE plate, the problem is
# symmetric under x->a-x and y->b-y (same as the uniform-load FFFF script),
# so peak deflection coincides with the geometric center -- unlike the
# triangular-load FFFF script, where it does not.
ix, iy = nx // 2, ny // 2
w_pred_center, w_anal_center = w_dim_pred[iy, ix], w_dim_anal[iy, ix]
center_abs_err = abs(w_pred_center - w_anal_center)
center_pct_err = center_abs_err / (abs(w_anal_center) + 1e-30) * 100

# Also report the true peak-deflection location for this asymmetric load.
peak_idx_anal = np.unravel_index(np.argmax(np.abs(w_dim_anal)), w_dim_anal.shape)
peak_idx_pred = np.unravel_index(np.argmax(np.abs(w_dim_pred)), w_dim_pred.shape)

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_anal) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_case": BC_CASE,
    "embedding_type": EMBEDDING_TYPE,
    "harmonic_indices": HARMONIC_INDICES,
    "use_product_terms": USE_PRODUCT_TERMS,
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_anal_m": float(w_anal_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "peak_deflection_anal_m": float(w_dim_anal[peak_idx_anal]),
    "peak_deflection_anal_xy_nd": [float(X_nd[peak_idx_anal]), float(Y_nd[peak_idx_anal])],
    "peak_deflection_pred_m": float(w_dim_pred[peak_idx_pred]),
    "peak_deflection_pred_xy_nd": [float(X_nd[peak_idx_pred]), float(Y_nd[peak_idx_pred])],
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": float(np.max(np.abs(w_dim_pred[0, :]))
                                         + np.max(np.abs(w_dim_pred[-1, :]))
                                         + np.max(np.abs(w_dim_pred[:, 0]))
                                         + np.max(np.abs(w_dim_pred[:, -1]))),
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_anal": w_dim_anal.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_anal": Mx_dim_anal.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_anal": My_dim_anal.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_anal": Mxy_dim_anal.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_anal"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_anal"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
plt.axhline(y=w_anal_center, linestyle="--", linewidth=2, color="k", label="FD reference")
plt.xlabel("Epoch"); plt.ylabel("Center Deflection w (m)")
plt.title("Convergence of Center Deflection")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_anal_center), linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total Loss (= PDE Loss)")
plt.plot(df_hist["epoch"], df_hist["hard_bc_residual_diagnostic"],
         label="Hard-BC deflection residual (diagnostic, ~0)", alpha=0.6, linestyle=":")
plt.plot(df_hist["epoch"], df_hist["slope_bc_residual_diagnostic"],
         label="Hard-BC slope residual (diagnostic, ~0)", alpha=0.6, linestyle=":")
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curve")
plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_anal, "FD Reference Plate Deflection (clamped, sinusoidal load)", "Deflection (m)",
             "Analytical_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_ref| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_ref", "w_pred - w_ref (m)",
             "Absolute_Error_w_pred_-_w_anal.png", cmap="coolwarm")

save_contour(Mx_dim_anal, "FD Reference Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x_Analytical.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_ref| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_anal, "FD Reference Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y_Analytical.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_ref| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_anal, "FD Reference Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy_Analytical.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_ref| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_anal.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_anal.min(), w_dim_pred.min()), max(w_dim_anal.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("FD reference displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("FD Reference vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Analytical_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}, {BC_CASE}")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_anal, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_anal, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_anal, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_anal, Mxy_dim_pred, Mxy_error),
]
for r, (label, anal, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"FD Reference {label}", anal, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading, {BC_CASE} (summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Total training time: {total_train_time:.2f}s | "
      f"Epochs to early-stop: {converged_epoch} | "
      f"Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Total training time: {total_train_time:.2f}s | Epochs: {converged_epoch}")

Epoch 0 Loss 2.663008e+00 (pde 2.663e+00) hardBC_residual 0.000e+00 slope_residual 0.000e+00 Center 4.009324e-04 (1842.63 ms/epoch)
Epoch 100 Loss 4.115317e-02 (pde 4.115e-02) hardBC_residual 0.000e+00 slope_residual 0.000e+00 Center 7.639116e-04 (1201.82 ms/epoch)
Epoch 200 Loss 3.763840e-02 (pde 3.764e-02) hardBC_residual 0.000e+00 slope_residual 0.000e+00 Center 7.708392e-04 (1201.45 ms/epoch)
Epoch 300 Loss 3.676079e-02 (pde 3.676e-02) hardBC_residual 0.000e+00 slope_residual 0.000e+00 Center 7.827117e-04 (1199.28 ms/epoch)
Epoch 400 Loss 3.473049e-02 (pde 3.473e-02) hardBC_residual 0.000e+00 slope_residual 0.000e+00 Center 7.917971e-04 (1208.20 ms/epoch)
Epoch 500 Loss 3.118067e-02 (pde 3.118e-02) hardBC_residual 0.000e+00 slope_residual 0.000e+00 Center 8.006650e-04 (1211.39 ms/epoch)
Epoch 600 Loss 3.008054e-02 (pde 3.008e-02) hardBC_residual 0.000e+00 slope_residual 0.000e+00 Center 8.099829e-04 (1214.70 ms/epoch)
Epoch 700 Loss 2.866398e-02 (pde 2.866e-02) hardBC_residual 0.00

C:\Users\vivek\AppData\Local\Temp\ipykernel_15620\43894488.py:554: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location=

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'DH_PINN_Modal_HardBC_FFFF_Sinusoidal', 'loading_case': 'sinusoidal', 'bc_case': 'FFFF', 'embedding_type': 'deterministic_harmonic_modal_full', 'harmonic_indices': [1, 2, 3, 4, 5], 'include_raw_coords': True, 'use_product_terms': True, 'grad_clip_norm': 1.0, 'final_pde_loss': 4.980046585740559e-06, 'final_hard_bc_residual_diagnostic': 0.0, 'final_slope_bc_residual_diagnostic': 0.0, 'note': 'total_loss == pde_loss for FFFF; both essential BCs are hard-enforced by the squared envelope, so there is no moment/soft-BC term to balance.'}
Saved cost_metrics.json: {'model': 'DH_PINN_Modal_HardBC_FFFF_Sinusoidal', 'loading_case': 'sinusoidal', 'bc_case': 'FFFF', 'embedding_type': 'deterministic_harmonic_modal_full', 'harmonic_indices': [1, 2, 3, 4, 5], 'include_raw_coords': True, 'use_product_terms': True, 'grad_clip_norm': 1.0, 'input_feature_dim': 47, 'device': 'cuda', 'epochs_run': 15000, 'epochs_to_early_stop': 15000, 'epochs